# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
import shap

import lightgbm as lgb

import xgboost as xgb

import umap
import hdbscan

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

import sqlite3
import os

#from flowmeter import convert_pcap_to_csv

/home/linkezio/Projects/IC-temp/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Constants

In [2]:
RANDOM_SEED = 42

# Inputs

In [3]:
DB_PATH = '../data/sqlite/data.db'

DATA_PATH = '../data/raw/CIC_IIoT_dataset_2025/generated_CSVs'

# Load Data

In [4]:
conn = sqlite3.connect(DB_PATH)

# Check available tables
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables in the database:")
for table in tables:
    print(f"  • {table[0]}")

Available tables in the database:
  • CIC_IIoT_dataset_2025
  • CIC_IoT_DIAD_2024
  • CIC_APT_IIoT_2024
  • CIC_BCCC_NRC_IoMT_2024
  • CIC_IoT_dataset_2023


In [5]:
def load_all_csvs_recursive(root_dir, pattern="*.csv", encoding=None, add_source_column=True, **read_csv_kwargs):
    """
    Load CSV files recursively from root_dir into a single DataFrame.

    Parameters
    ----------
    root_dir : str or Path
        Root directory to search recursively.
    pattern : str
        Filename glob pattern (default: "*.csv").
    encoding : str | None
        Optional encoding for pandas.read_csv.
    add_source_column : bool
        If True, adds a `__source_file__` column with file path.
    **read_csv_kwargs : dict
        Extra kwargs forwarded to pandas.read_csv.

    Returns
    -------
    pd.DataFrame
        Concatenated DataFrame from all found CSVs.
    """
    import fnmatch

    root_dir = os.path.abspath(root_dir)

    csv_paths = []
    for current_root, _, files in os.walk(root_dir):
        for file_name in files:
            if fnmatch.fnmatch(file_name, pattern):
                csv_paths.append(os.path.join(current_root, file_name))

    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found recursively in: {root_dir} (pattern: {pattern})")

    frames = []
    failures = []

    for path in sorted(csv_paths):
        try:
            kwargs = dict(read_csv_kwargs)
            if encoding is not None:
                kwargs["encoding"] = encoding

            df_part = pd.read_csv(path, **kwargs)

            if add_source_column:
                df_part["__source_file__"] = path

            frames.append(df_part)
        except Exception as exc:
            failures.append((path, str(exc)))

    if not frames:
        raise ValueError("All CSV files failed to load.")

    merged_df = pd.concat(frames, ignore_index=True, sort=False)

    print(f"Loaded {len(frames)} CSV file(s)")
    print(f"Merged shape: {merged_df.shape}")

    if failures:
        print(f"Failed to load {len(failures)} file(s):")
        for file_path, err in failures[:10]:
            print(f"  - {file_path}: {err}")
        if len(failures) > 10:
            print(f"  ... and {len(failures) - 10} more")

    return merged_df


# Phase 1 - Binary Classification (Benign vs Malicious)

In [6]:
df = pd.read_sql_query("SELECT * FROM CIC_IIoT_dataset_2025", conn)

In [7]:
df

,device_name,device_mac,label_full,label1,label2,label3,label4,timestamp,timestamp_start,timestamp_end,...,network_time-delta_min,network_time-delta_std_deviation,network_ttl_avg,network_ttl_max,network_ttl_min,network_ttl_std_deviation,network_window-size_avg,network_window-size_max,network_window-size_min,network_window-size_std_deviation
0,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:10.709000Z_2025-01-23T15:31:1...,2025-01-23T15:31:10.709000Z,2025-01-23T15:31:12.709000Z,...,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
1,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:12.709000Z_2025-01-23T15:31:1...,2025-01-23T15:31:12.709000Z,2025-01-23T15:31:14.709000Z,...,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
2,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:14.709000Z_2025-01-23T15:31:1...,2025-01-23T15:31:14.709000Z,2025-01-23T15:31:16.709000Z,...,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
3,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:16.709000Z_2025-01-23T15:31:1...,2025-01-23T15:31:16.709000Z,2025-01-23T15:31:18.709000Z,...,0.000000e+00,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000
4,edge1,dc:a6:32:dc:27:d4,attack_ddos_syn-flood-port-80_edge1,attack,ddos,syn-flood-port-80,ddos_syn-flood-port-80,2025-01-23T15:31:18.709000Z_2025-01-23T15:31:2...,2025-01-23T15:31:18.709000Z,2025-01-23T15:31:20.709000Z,...,2.600000e-08,0.000042,64.0,64.0,64.0,0.0,17587.532313,64240.0,0.0,28377.701703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
685666,plug-all-sensors,d4:a6:51:82:98:a8,benign_whole-network3,benign,benign,benign,benign,2025-09-09T15:08:50.400000Z_2025-09-09T15:09:0...,2025-09-09T15:08:50.400000Z,2025-09-09T15:09:00.400000Z,...,1.555000e-03,0.015271,255.0,255.0,255.0,0.0,0.000000,0.0,0.0,0.000000
685667,plug-all-sensors,d4:a6:51:82:98:a8,benign_whole-network3,benign,benign,benign,benign,2025-09-09T15:09:00.400000Z_2025-09-09T15:09:1...,2025-09-09T15:09:00.400000Z,2025-09-09T15:09:10.400000Z,...,2.296000e-03,0.013650,255.0,255.0,255.0,0.0,0.000000,0.0,0.0,0.000000
685668,plug-all-sensors,d4:a6:51:82:98:a8,benign_whole-network3,benign,benign,benign,benign,2025-09-09T15:09:10.400000Z_2025-09-09T15:09:2...,2025-09-09T15:09:10.400000Z,2025-09-09T15:09:20.400000Z,...,4.307000e-03,0.024962,255.0,255.0,255.0,0.0,0.000000,0.0,0.0,0.000000
685669,plug-all-sensors,d4:a6:51:82:98:a8,benign_whole-network3,benign,benign,benign,benign,2025-09-09T15:09:20.400000Z_2025-09-09T15:09:3...,2025-09-09T15:09:20.400000Z,2025-09-09T15:09:30.400000Z,...,8.460000e-04,0.003789,255.0,255.0,255.0,0.0,0.000000,0.0,0.0,0.000000


In [8]:
y = df['label1']
X = df.drop(columns=df.select_dtypes(exclude='number').columns)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

A prioridade aqui é o recall, para diminuir falsos negativos, porque é melhor suspeitar de dados positivos do que deixar passar dados negativos como benignos.

In [9]:
def objective(trial):
    threshold = trial.suggest_float("threshold", 0.01, 0.5)

    params = {
        "objective": "multiclass",
        "num_class": y_train.nunique(),
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "random_state": RANDOM_SEED,

        # 🌳 Estrutura da árvore (mais importante)
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 16),

        # 🎯 Controle de overfitting
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),

        # 🚀 Learning
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),

        # 🎲 Amostragem (muito importante pra generalização)
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),

        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        # 🧱 Regularização
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),

        # ⚡ Feature binning (impacta MUITO performance)
        "max_bin": trial.suggest_int("max_bin", 128, 512),

        # 🧠 IMPORTANTE pra reduzir FN
        "class_weight": "balanced",

        # ⚙️ Sistema
        "verbosity": -1,
        "force_row_wise": True,
        "n_jobs": 3,  # melhor deixar 1 e paralelizar no CV
    }

    model = lgb.LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)

        probs = model.predict_proba(X_val)

        # índice da classe "attack"
        attack_idx = list(model.classes_).index("attack")

        y_pred = (probs[:, attack_idx] > threshold).astype(int)
        y_val_bin = (y_val == "attack").astype(int)

        score = recall_score(y_val_bin, y_pred)
        scores.append(score)

    return np.mean(scores)

In [10]:

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, timeout=1800, show_progress_bar=True)

[I 2026-04-26 15:51:16,424] A new study created in memory with name: no-name-b912b1b4-6e47-4848-96d8-ce461b3f4ace
Best trial: 0. Best value: 0.997338:   5%|▌         | 1/20 [00:53<16:48, 53.07s/it, 53.07/1800 seconds]

[I 2026-04-26 15:52:09,500] Trial 0 finished with value: 0.9973383727619005 and parameters: {'threshold': 0.05898305232677746, 'num_leaves': 58, 'max_depth': 16, 'min_child_samples': 54, 'min_split_gain': 0.04930878706758912, 'learning_rate': 0.03163450716134615, 'n_estimators': 501, 'subsample': 0.7076268516974892, 'subsample_freq': 1, 'colsample_bytree': 0.8758348299787788, 'reg_alpha': 0.114716005837944, 'reg_lambda': 0.0006794567468708971, 'max_bin': 187}. Best is trial 0 with value: 0.9973383727619005.


Best trial: 0. Best value: 0.997338:  10%|█         | 2/20 [01:44<15:37, 52.06s/it, 104.42/1800 seconds]

[I 2026-04-26 15:53:00,850] Trial 1 finished with value: 0.9312668852190944 and parameters: {'threshold': 0.2192389948322951, 'num_leaves': 108, 'max_depth': 8, 'min_child_samples': 12, 'min_split_gain': 0.15218983193499358, 'learning_rate': 0.08959475814531917, 'n_estimators': 614, 'subsample': 0.9250101608605394, 'subsample_freq': 7, 'colsample_bytree': 0.8792750039012885, 'reg_alpha': 2.7271047232001577e-06, 'reg_lambda': 1.0385173911910067e-06, 'max_bin': 175}. Best is trial 0 with value: 0.9973383727619005.


Best trial: 0. Best value: 0.997338:  15%|█▌        | 3/20 [02:33<14:21, 50.66s/it, 153.43/1800 seconds]

...

[I 2026-04-26 16:06:13,729] Trial 17 finished with value: 0.9950801549434821 and parameters: {'threshold': 0.07711299449943737, 'num_leaves': 127, 'max_depth': 6, 'min_child_samples': 20, 'min_split_gain': 0.7067404435616125, 'learning_rate': 0.04340592560635059, 'n_estimators': 686, 'subsample': 0.6695790223123306, 'subsample_freq': 4, 'colsample_bytree': 0.6804701889168893, 'reg_alpha': 0.6197941516994755, 'reg_lambda': 3.6361376745472795, 'max_bin': 308}. Best is trial 12 with value: 0.9999386116652226.


Best trial: 12. Best value: 0.999939:  95%|█████████▌| 19/20 [15:53<00:53, 53.47s/it, 953.39/1800 seconds]

[I 2026-04-26 16:07:09,813] Trial 18 finished with value: 0.9847405903760239 and parameters: {'threshold': 0.17123967464328832, 'num_leaves': 68, 'max_depth': 10, 'min_child_samples': 42, 'min_split_gain': 0.4323031208134796, 'learning_rate': 0.005683155944484503, 'n_estimators': 431, 'subsample': 0.8106369536923161, 'subsample_freq': 5, 'colsample_bytree': 0.7383477384012513, 'reg_alpha': 0.0061542537070164785, 'reg_lambda': 6.0348117399712346e-05, 'max_bin': 223}. Best is trial 12 with value: 0.9999386116652226.


Best trial: 12. Best value: 0.999939: 100%|██████████| 20/20 [16:50<00:00, 50.55s/it, 1010.91/1800 seconds]

[I 2026-04-26 16:08:07,335] Trial 19 finished with value: 0.9256366882614603 and parameters: {'threshold': 0.32741403929981105, 'num_leaves': 141, 'max_depth': 6, 'min_child_samples': 19, 'min_split_gain': 0.2592920973526145, 'learning_rate': 0.09934233677084435, 'n_estimators': 715, 'subsample': 0.8770074381238596, 'subsample_freq': 4, 'colsample_bytree': 0.654342331054066, 'reg_alpha': 0.00031293162848677707, 'reg_lambda': 0.34086859304194084, 'max_bin': 219}. Best is trial 12 with value: 0.9999386116652226.


In [11]:
print("Best score:", study.best_value)
print("Best params:", study.best_params)

Best score: 0.9999386116652226
Best params: {'threshold': 0.010925906410474542, 'num_leaves': 31, 'max_depth': 16, 'min_child_samples': 22, 'min_split_gain': 0.29382617039480574, 'learning_rate': 0.08990687983517043, 'n_estimators': 545, 'subsample': 0.6854237740541148, 'subsample_freq': 3, 'colsample_bytree': 0.6033855536816999, 'reg_alpha': 9.85338159849272, 'reg_lambda': 0.1531019875935439, 'max_bin': 133}


In [12]:
model = lgb.LGBMClassifier(**study.best_params)

In [13]:
model.fit(X_train, y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,16
,learning_rate,0.08990687983517043
,n_estimators,545
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.29382617039480574
,min_child_weight,0.001
,min_child_samples,22


In [14]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

      attack       1.00      0.92      0.95     56943
      benign       0.94      1.00      0.97     80192

    accuracy                           0.96    137135
   macro avg       0.97      0.96      0.96    137135
weighted avg       0.96      0.96      0.96    137135



In [17]:
recall_score(y_test, y_pred, pos_label="benign")

0.997032122905028

In [ ]:
probs = model.predict_proba(X_test)
probs

array([[ True, False],
       [False,  True],
       [False,  True],
       ...,
       [False,  True],
       [False,  True],
       [False,  True]], shape=(137135, 2))

# Phase 2 - Multiclassification

# Phase 3 - Clustering